In [7]:
import cv2
import time
import mediapipe as mp
import os
import sys

CURRENT_DIR = os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(CURRENT_DIR, "../../"))

if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
    
YOLO_MODEL_PATH = os.path.join(ROOT_DIR, "models", "yolov8x.pt")
MP_MODEL_PATH = os.path.join(ROOT_DIR, "models", "gesture_recognizer.task")

# Thiết lập các Alias
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
GestureRecognizerResult = mp.tasks.vision.GestureRecognizerResult
VisionRunningMode = mp.tasks.vision.RunningMode

# Biến toàn cục
latest_gesture = "None"
current_landmarks_list = []
# --- BIẾN ĐO FPS ---
prev_frame_time = 0
new_frame_time = 0

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)
]

def print_result(result: GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int):
    global latest_gesture, current_landmarks_list
    if result.gestures and len(result.gestures) > 0:
        latest_gesture = result.gestures[0][0].category_name
    else:
        latest_gesture = "None"
    
    current_landmarks_list = result.hand_landmarks if result.hand_landmarks else []

def main():
    global prev_frame_time, new_frame_time
    
    options = GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=MP_MODEL_PATH),
        running_mode=VisionRunningMode.LIVE_STREAM,
        num_hands=2,
        result_callback=print_result
    )

    with GestureRecognizer.create_from_options(options) as recognizer:
        
        # --- CHỌN NGUỒN VIDEO ---
        # Cách 1: Webcam thật (thường là 0 hoặc 1)
        # cap = cv2.VideoCapture(0) 
        
        # Cách 2: iVCam qua RTSP (Thay IP của máy bạn vào đây)
        # cap = cv2.VideoCapture("rtsp://192.168.1.XX:4747/live")
        
        cap = cv2.VideoCapture(1) # Đang để mặc định theo code của bạn
        
        print("Nhấn 'q' để thoát.")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            # --- TÍNH TOÁN FPS ---
            new_frame_time = time.time()
            # fps = 1 / (thời gian khung hình hiện tại - thời gian khung hình trước)
            fps = 1 / (new_frame_time - prev_frame_time) if (new_frame_time - prev_frame_time) > 0 else 0
            prev_frame_time = new_frame_time
            fps = int(fps)

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(time.time() * 1000)

            recognizer.recognize_async(mp_image, timestamp_ms)

            # Vẽ Landmarks
            if current_landmarks_list:
                h, w, _ = frame.shape
                for hand_landmarks in current_landmarks_list:
                    for connection in HAND_CONNECTIONS:
                        start_idx, end_idx = connection
                        sx, sy = int(hand_landmarks[start_idx].x * w), int(hand_landmarks[start_idx].y * h)
                        ex, ey = int(hand_landmarks[end_idx].x * w), int(hand_landmarks[end_idx].y * h)
                        cv2.line(frame, (sx, sy), (ex, ey), (255, 0, 0), 2)
                    for landmark in hand_landmarks:
                        cx, cy = int(landmark.x * w), int(landmark.y * h)
                        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

            # --- HIỂN THỊ FPS VÀ GESTURE ---
            cv2.putText(frame, f'FPS: {fps}', (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
            cv2.putText(frame, f'Gesture: {latest_gesture}', (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow('Real-time Gesture & Landmarks', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break

        cap.release()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()

Nhấn 'q' để thoát.
